# RoadGuard — Model Training and Evaluation

Run one LOFO fold: RGB+depth teacher, RGB baseline and distilled RGB student. Training, selection and evaluation live in `roadguard`; everything that varies between runs lives in a YAML under `configs/`.

## 1. Mount Drive and install RoadGuard

Use a GPU Colab runtime. Clone the repository if needed, or pull the latest commit before proceeding. SegFormer needs the `transformers` version pinned by the repository; do not upgrade packages in this notebook.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os
import subprocess
import base64

# Mount Google Drive
drive.mount("/content/drive")

# Clone private RoadGuard repository
REPO = Path("/content/road-guard")

if not REPO.exists():
    token = userdata.get("GITHUB_TOKEN")

    credentials = base64.b64encode(
        f"x-access-token:{token}".encode()
    ).decode()

    env = os.environ.copy()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {credentials}"

    subprocess.run(
        ["git", "clone", "https://github.com/RanaShoaaib/RoadGuard.git", str(REPO)],
        check=True,
        env=env,
    )

    del token, credentials, env

# Install RoadGuard
os.chdir(REPO)
subprocess.run(["python", "-m", "pip", "install", "-e", "."], check=True)

print("Repository:", REPO)

In [ ]:
import sys
sys.path.insert(0, str(REPO / "src"))

# Verify the import before starting training.
from roadguard.training.experiment import run_experiment
print("RoadGuard imported successfully")

## 2. Stage the prepared dataset locally

Notebook 1 saved one TAR package per archive in `RoadGuard_Processed/towns/` and the manifest and LOFO splits in `RoadGuard_Processed/processed/`. Unpacking them to the Colab disk avoids repeated Google Drive reads during training. Allow sufficient local disk space for the unpacked files.

In [ ]:
from roadguard.data.preparation import stage_prepared_dataset

DRIVE_PREPARED = Path("/content/drive/MyDrive/RoadGuard_Processed")
STAGE = Path("/content/roadguard_data")

staged = stage_prepared_dataset(DRIVE_PREPARED, STAGE)
DATA_ROOT = staged["data_root"]
LOFO_DIR = staged["processed_dir"] / "lofo_splits"
print("Dataset staged:", DATA_ROOT)
print("LOFO folds:", sorted(p.name for p in LOFO_DIR.glob("*.csv")))


## 3. Run the configured experiment

The YAML selects architecture, held-out family, seed, epochs and the training recipe (scenario-balanced sampling, focal+Dice loss, EMA weights, cosine schedule, smoothed validation-AP checkpoint selection). `experiment.roles` chooses which models to train (e.g. `[teacher, baseline]` to check the teacher before training a student). The runner refuses to overwrite a nonempty run directory, and an interrupted run cannot resume.

In [ ]:
import torch
from roadguard.training.config import load_config
from roadguard.training.experiment import run_experiment

# EXPERIMENT CONFIGURATION
# Pick the architecture file, then override anything else here (or edit the YAML).
CONFIG_PATH = REPO / "configs" / "unet.yaml"      # unet.yaml | segformer.yaml
FAMILY = "special"                                # animal | home | special
SEED = 42

OUTPUT_ROOT = Path("/content/drive/MyDrive/RoadGuard_Experiments")
config = load_config(CONFIG_PATH)
config["experiment"].update(lofo_family=FAMILY, seed=SEED)
FOLD_PATH = LOFO_DIR / f"{FAMILY}.csv"
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training")
print("GPU:", torch.cuda.get_device_name(0))
print("Experiment:", config["experiment"])

results = run_experiment(config, FOLD_PATH, DATA_ROOT, OUTPUT_ROOT, device="cuda")


## 4. Compare held-out test performance

The primary metric is **macro scenario AP**: average precision computed within each held-out scenario, then averaged, so large scenarios do not dominate. Dice, precision and recall use a threshold chosen on validation and applied once to test. *Selected* is the checkpoint picked by smoothed validation AP; *last* is the final epoch.

In [ ]:
import pandas as pd
from roadguard.evaluation.reports import load_reports

table = load_reports(OUTPUT_ROOT, [config["experiment"]["name"]], FAMILY, [SEED],
                     roles=config["experiment"]["roles"])
display(table.drop(columns=["run", "family", "seed"]).round(4))
print("Saved experiment results:", OUTPUT_ROOT / config["experiment"]["name"] / FAMILY / f"seed_{SEED}")
